<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/exams/midterm/MID2_grounding_naming_game.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Midterm Capstone: Machines, Tests and Meaning

## Project 2: Grounding through language games

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

A population of agents invents and aligns a vocabulary for colours, sizes and shapes by playing naming games about objects they perceive. The vocabulary is then studied as text only: Co-occurrence recovers the dimensions of meaning, but reference stays at chance unless some words are grounded [1, 2].

**Required extensions.** Add perceptual noise and continuous features so that categories must be learned rather than given, compare populations of different sizes, and test whether structural information can propagate reference from anchor words to others. Discuss what the results imply for conceptual role semantics and for the vector grounding debate [3, 4].

## What this example implements

The notebook implements a naming game with lateral inhibition, plots the rise of communicative success, builds a corpus from the evolved vocabulary, recovers its dimensions by spectral clustering of co-occurrence and measures reference accuracy as grounded anchor words are added.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. A naming game

In [ ]:
import itertools
rng = np.random.default_rng(SEED)
DIMS = {"colour": ["red", "green", "blue", "yellow", "purple", "orange"], "size": ["small", "medium", "large"], "shape": ["round", "square", "long"]}
DIM_OF = {v: d for d, vs in DIMS.items() for v in vs}
OBJECTS = list(itertools.product(*DIMS.values()))

def new_word():
    return "".join(str(rng.choice(list("bdgklmnprstvz"))) + str(rng.choice(list("aeiou"))) for _ in range(2))

class Agent:
    def __init__(self):
        self.lex = {}
    def word_for(self, value):
        cands = [(s, w) for (w, v), s in self.lex.items() if v == value]
        if not cands:
            w = new_word(); self.lex[(w, value)] = 0.5
            return w
        return max(cands)[1]
    def interpret(self, word, context):
        for s, v in sorted(((s, v) for (w, v), s in self.lex.items() if w == word), reverse=True):
            hits = [o for o in context if v in o]
            if len(hits) == 1:
                return hits[0], v
        return None, None

def play(pop, rounds=8000, ctx_size=4):
    success = []
    for _ in range(rounds):
        i, j = rng.choice(len(pop), 2, replace=False)
        S, Hr = pop[i], pop[j]
        ctx = [OBJECTS[k] for k in rng.choice(len(OBJECTS), ctx_size, replace=False)]
        topic = ctx[0]
        disc = [v for v in topic if sum(v in o for o in ctx) == 1]
        if not disc:
            continue
        value = disc[rng.integers(len(disc))]
        w = S.word_for(value)
        guess, v = Hr.interpret(w, ctx)
        if guess == topic:
            for ag, val in ((S, value), (Hr, v)):
                ag.lex[(w, val)] = min(1.0, ag.lex[(w, val)] + 0.1)
                for key in [k for k in ag.lex if k[1] == val and k[0] != w]:
                    ag.lex[key] -= 0.1
            success.append(1.0)
        else:
            S.lex[(w, value)] -= 0.1
            Hr.lex.setdefault((w, value), 0.5)
            success.append(0.0)
    return np.array(success)

population = [Agent() for _ in range(10)]
succ = play(population)
plt.plot(np.convolve(succ, np.ones(200) / 200, mode="valid")); plt.xlabel("game"); plt.ylabel("communicative success"); plt.title("A grounded naming game"); plt.show()
print("final success rate:", succ[-500:].mean().round(3))

## 2. Structure from text, reference from grounding

In [ ]:
speaker = population[0]
names = {val: speaker.word_for(val) for val in DIM_OF}
sentences = [[names[v] for v in OBJECTS[k]] for k in rng.integers(0, len(OBJECTS), 3000)]
vocab = sorted(set(names.values()))
ix = {w: i for i, w in enumerate(vocab)}
C = np.zeros((len(vocab), len(vocab)))
for s in sentences:
    for a in s:
        for b in s:
            if a != b:
                C[ix[a], ix[b]] += 1
never = (C == 0).astype(float)
np.fill_diagonal(never, 0)
from sklearn.cluster import SpectralClustering
labels = SpectralClustering(n_clusters=3, affinity="precomputed", random_state=SEED).fit_predict(never + 1e-6)
true_dim = [DIM_OF[next(v for v, w in names.items() if w == word)] for word in vocab]
purity = sum(max(np.sum((labels == c) & (np.array(true_dim) == d)) for d in DIMS) for c in range(3)) / len(vocab)
print(f"distribution alone recovers the three dimensions with purity {purity:.2f}")

def reference_accuracy(n_anchors, trials=3000):
    anchors = set(rng.choice(list(DIM_OF), n_anchors, replace=False)) if n_anchors else set()
    hits = 0
    for _ in range(trials):
        ctx = [OBJECTS[k] for k in rng.choice(len(OBJECTS), 4, replace=False)]
        disc = [v for v in ctx[0] if sum(v in o for o in ctx) == 1]
        if not disc:
            hits += 0.25; continue
        v = disc[rng.integers(len(disc))]
        hits += 1.0 if v in anchors else 0.25
    return hits / trials

xs = list(range(0, 13, 2))
plt.plot(xs, [reference_accuracy(k) for k in xs], "o-"); plt.axhline(0.25, ls="--", color="k")
plt.xlabel("number of grounded anchor words"); plt.ylabel("reference accuracy of a distributional hearer"); plt.show()

## Report and submission

Both tracks follow the course report template. A research report states its thesis in the introduction, reconstructs the relevant arguments as explicit premises and conclusions, and evaluates them. An application report explains what the simulation or experiment can and cannot show about the philosophical question, presents its results with figures and discusses its limitations. All sources are cited in square brackets.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the midterm capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 7, with the report and all code files attached or linked.

## References

[1] Harnad, S. (1990). The symbol grounding problem. *Physica D: Nonlinear Phenomena*, 42(1-3), 335-346. <https://doi.org/10.1016/0167-2789(90)90087-6>

[2] Piantadosi, S. T., & Hill, F. (2022). Meaning without reference in large language models. arXiv preprint arXiv:2208.02957. <https://arxiv.org/abs/2208.02957>

[3] Mollo, D. C., & Millière, R. (2023). The vector grounding problem. arXiv preprint arXiv:2304.01481. <https://arxiv.org/abs/2304.01481>

[4] Bender, E. M., & Koller, A. (2020). Climbing towards NLU: On meaning, form, and understanding in the age of data. In *Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics* (pp. 5185-5198). <https://doi.org/10.18653/v1/2020.acl-main.463>